<a href="https://colab.research.google.com/github/juhong7586/livekit-ai-toy-model/blob/main/ai_toy_wakeword.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Summary of the result

1) Hey Kiki: AUT=0.0000  FPPH=0.00  Recall=97.0%

2) OK Kiki: AUT=0.0000  FPPH=0.00  Recall=94.7%

3) Kiki: AUT=0.0000  FPPH=0.00  Recall=99.9%

4) Wiki: AUT=0.0000  FPPH=0.00  Recall=99.9%


In [1]:
# 1. Install
!pip install git+https://github.com/livekit/livekit-wakeword.git
!apt-get install -y espeak-ng ffmpeg

  Cloning https://github.com/livekit/livekit-wakeword.git to /tmp/pip-req-build-_135ivt2
  Running command git clone --filter=blob:none --quiet https://github.com/livekit/livekit-wakeword.git /tmp/pip-req-build-_135ivt2
  Resolved https://github.com/livekit/livekit-wakeword.git to commit 00502c4685bd6722431b321450bae05ee567a30d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 117.3 MB/s eta 0:00:00
  Created wheel for livekit-wakeword: filename=livekit_wakeword-0.1.4.dev1+g00502c468-py3-none-any.whl size=1901642 sha256=c979b3208def6648423eaabe73a6822816bb1e4ee6e69f3111019ef253ac5add
  Stored in directory: /tmp/pip-ephem-wheel-cache-3ol8clcv/wheels/67/33/50/e7feadf35342d1ea4a0d7fa59a5cdc2097525a062dc64c2ad8
Successfully built livekit-wakeword
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg i

In [2]:
!pip install onnx onnxruntime torch webrtcvad pronouncing audiomentations onnxscript

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 7.3 MB/s eta 0:00:00
  Created wheel for webrtcvad: filename=webrtcvad-2.0.10-cp312-cp312-linux_x86_64.whl size=73525 sha256=c590251599d052c3f3400f327c84a229a23163a410983088a9cf3f4f0816f229
  Stored in directory: /root/.cache/pip/wheels/1e/d3/95/680fa3b16848f1a58d2edaed34c496224c89a9bc63e17b3614

In [3]:
# 2. Set up (Download the model)
!python -m livekit.wakeword setup

Streaming output truncated to the last 5000 lines.
                             se/free-sound/noise-free-sound-0158                
                             .wav "HTTP/1.1 302 Found"                          

                             https://huggingface.co/datasets/Flu                
                             idInference/musan/resolve/3edcfdf89                
                             b56dbe6a395ff29f9c29489e03d1321/noi                
                             se/free-sound/noise-free-sound-0159                
                             .wav "HTTP/1.1 302 Found"                          

                             https://huggingface.co/datasets/Flu                
                             idInference/musan/resolve/3edcfdf89                
                             b56dbe6a395ff29f9c29489e03d1321/noi                
                             se/free-sound/noise-free-sound-0160                
                             .wav "HTTP/1.1 302 Found"  

# Hey kiki

In [ ]:
# 3. Training
from livekit.wakeword import WakeWordConfig, run_generate, run_augment, run_extraction, run_train, run_export, run_eval
positive_phrases = [
    "hey kiki",
    "hey, kiki",
    "hey kiki!",
    "hey kiki?",
    "hey kiki.",
    "heykiki",
    "hey kiki hey kiki",
    "hey... kiki",
    "hey  kiki",
]

negative_phrases = [
    "hey kiwi",
    "hey wiki",
    "hey mickey",
    "hey ricky",
    "hey kitty",
    "hey kicky",
    "hey gigi",
    "hey we key",
    "hey quick",
    "hey keep it",
    "hey kid",
    "hey give me",
    "hey okay",
    "hey keko",
    "hey kiko",
    "hey kaki",
    "hey kiku",
    "hey keka",
    "hey kitty cat",
    "hey kiwi fruit",
    "hey kick it",
]

config_heykiki = WakeWordConfig(
    model_name="HeyKiki",
    target_phrases=["hey kiki"],
    n_samples=12000,
    n_samples_val=2000,
    model_size="small",
    steps=30000,
    custom_negative_phrases=negative_phrases,
    custom_positive_phrases=positive_phrases
)

In [ ]:
# Run individual stages
run_generate(config_heykiki)     # TTS synthesis + adversarial negatives

In [ ]:
run_augment(config_heykiki)

In [ ]:
run_extraction(config_heykiki)

In [ ]:
run_train(config_heykiki)

In [ ]:
onnx_path = run_export(config_heykiki)

In [ ]:
results = run_eval(config_heykiki, onnx_path)
print(f"AUT={results['aut']:.4f}  FPPH={results['fpph']:.2f}  Recall={results['recall']:.1%}")

# Kiki

In [ ]:
negative_phases = ["kiwi", "miki", "tiki", "kitty", "kicky", "gigi"]

config_kiki = WakeWordConfig(
model_name="kiki",
target_phrases=["kiki"],
n_samples=12000,
n_samples_val=2000,
model_size="small",
steps=30000,
custom_negative_phrases=negative_phases
)

In [ ]:
run_generate(config_kiki)

In [ ]:
run_augment(config_kiki)

In [ ]:
run_extraction(config_kiki)   # Extract mel spectrograms + speech embeddings → .npy

In [ ]:
run_train(config_kiki)        # 3-phase adaptive training

In [ ]:
onnx_path = run_export(config_kiki)       # Export to ONNX

In [ ]:
# Evaluate the exported model
results = run_eval(config_kiki, onnx_path)
print(f"AUT={results['aut']:.4f}  FPPH={results['fpph']:.2f}  Recall={results['recall']:.1%}")

# Okay Kiki

In [ ]:
positive_phrases = [
    "okay kiki",
    "ok kiki",
    "okay, kiki",
    "okay kiki!",
    "okay kiki?",
    "okay kiki.",
    "okaykiki",
    "ok kiki",
    "ok, kiki",
    "ok kiki!",
    "okkiki",
    "okay kiki okay kiki",
    "okay... kiki",
    "okay  kiki",
]

negative_phrases = [
    "okay kiwi",
    "okay wiki",
    "okay mickey",
    "okay ricky",
    "okay kitty",
    "okay gigi",
    "okay keko",
    "okay kiko",
    "okay kaki",
    "okay kiku",
    "okay keka",
    "okay key key",
    "okay we key",
    "okay quick",
    "okay keep it",
    "okay kid",
    "okay give me",
    "ok kiwi",
    "ok wiki",
    "ok mickey",
    "ok ricky",
    "ok kitty",
    "ok kicky",
    "ok gigi",
    "ok keko",
    "ok kiko",
    "ok kaki",
    "ok kiku",
    "ok keka",
    "ok key key",
    "ok we key",
    "ok quick",
    "ok keep it",
    "ok kid",
    "ok give me",
    "ok coco",
    "okay kick it",
]

config_okkiki = WakeWordConfig(
model_name="okKiki",
target_phrases=["ok kiki"],
n_samples=12000,
n_samples_val=2000,
model_size="small",
steps=30000,
custom_negative_phrases=negative_phases,
custom_positive_phrases=positive_phrases
)

In [ ]:
run_generate(config_okkiki)

In [ ]:
run_augment(config_okkiki)

In [ ]:
run_extraction(config_okkiki)

In [ ]:
run_train(config_okkiki)

In [ ]:
onnx_path = run_export(config_okkiki)

In [ ]:
results = run_eval(config_okkiki, onnx_path)
print(f"AUT={results['aut']:.4f}  FPPH={results['fpph']:.2f}  Recall={results['recall']:.1%}")

# Wiki

In [6]:
from livekit.wakeword import WakeWordConfig, run_generate, run_augment, run_extraction, run_train, run_export, run_eval

positive_phrases = [
    "wiki",
    "wiki!",
    "wiki?",
    "wiki.",
    "wiki wiki",
    "wiki... wiki",
]
negative_phrases = [
    "kiki",
    "kiwi",
    "mickey",
    "ricky",
    "tiki",
    "picky",
    "wicked",
    "wicket",
    "weekly",
    "why key",
    "wikid",
    "wiki page",
    "wikipedia",
]

config_wiki = WakeWordConfig(
model_name="wiki",
target_phrases=["wiki"],
n_samples=12000,
n_samples_val=2000,
model_size="small",
steps=30000,
custom_positive_phrases=positive_phrases,
custom_negative_phrases=negative_phrases
)

In [7]:
run_generate(config_wiki)

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...


Synthesizing clips: 100%|██████████| 12000/12000 [02:28<00:00, 80.90clip/s]
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...


Synthesizing clips: 100%|██████████| 2000/2000 [00:23<00:00, 83.39clip/s]


Removing weight norm...


Synthesizing clips: 100%|██████████| 12000/12000 [02:42<00:00, 73.78clip/s]


Removing weight norm...


Background (background_test): 100%|██████████| 40/40 [00:00<00:00, 290.19clip/s]


In [8]:
run_augment(config_wiki)

Augmenting background_test r0: 100%|██████████| 40/40 [00:00<00:00, 131.32clip/s]


In [9]:
run_extraction(config_wiki)

Features background_test: 100%|██████████| 40/40 [00:00<00:00, 103.98clip/s]


In [10]:
run_train(config_wiki)

Phase 1:  75%|███████▌  | 22510/30000 [07:22<03:26, 36.23step/s, loss=0.2035, lr=5.5e-05, neg_w=1126]

  Validation @ step 22500: FPPH=0.00, Recall=0.996, Acc=0.998


Phase 1:  80%|████████  | 24010/30000 [07:50<02:37, 38.04step/s, loss=6.7214, lr=3.9e-05, neg_w=1201]

  Validation @ step 24000: FPPH=0.00, Recall=0.998, Acc=0.999


Phase 1:  85%|████████▌ | 25509/30000 [08:18<01:59, 37.59step/s, loss=3.3813, lr=2.3e-05, neg_w=1276]

...

  Validation @ step 2700: FPPH=0.00, Recall=0.999, Acc=1.000


Phase 3:  95%|█████████▌| 2858/3000 [00:56<00:03, 37.81step/s, loss=9.6535, lr=5.4e-09, neg_w=1430]

  Validation @ step 2850: FPPH=0.00, Recall=0.999, Acc=1.000


Phase 3: 100%|██████████| 3000/3000 [00:59<00:00, 50.68step/s, loss=0.1703, lr=2.7e-13, neg_w=1500]


PosixPath('output/wiki/wiki.pt')

In [11]:
onnx_path = run_export(config_wiki)

/usr/local/lib/python3.12/dist-packages/livekit/wakeword/export/onnx.py:35: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `WakeWordClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `WakeWordClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 6 of general pattern rewrite rules.


In [12]:
results = run_eval(config_wiki, onnx_path)
print(f"AUT={results['aut']:.4f}  FPPH={results['fpph']:.2f}  Recall={results['recall']:.1%}")

AUT=0.0000  FPPH=0.00  Recall=99.9%
